# Ноутбук 9 — Эксперимент 6: Попытка глубокого обучения (LSTM / BiLSTM)

**Назначение.** Проверка гипотезы «LSTM/BiLSTM на временных последовательностях
окон превзойдут Random Forest». Архитектуры: LSTM(128) и BiLSTM(128) на
последовательностях K=10 окон, валидация LOSO (32 фолда).

In [1]:
import sys, time
from pathlib import Path

NB_ROOT = Path.cwd()
if str(NB_ROOT) not in sys.path:
    sys.path.insert(0, str(NB_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from modules import config
from modules.experiments import (
    NON_FEATURE_COLS, add_per_subject_target, per_subject_zscore,
)
from modules.deep import build_sequences, train_fold, K_DEFAULT
from modules.validation import compute_classification_metrics

print(f"Notebooks root: {NB_ROOT}")
print(f"PyTorch: {torch.__version__}")

Notebooks root: D:\Programming\Python\Diplom\notebooks
PyTorch: 2.11.0+cpu


## 9.1 Подготовка последовательностей

In [2]:
df = pd.read_csv(config.RESULTS_DIR / "feature_table.csv")
df = add_per_subject_target(df)
feat_cols = [c for c in df.columns if c not in NON_FEATURE_COLS
             and c != "arousal_class_persubj"]

X = per_subject_zscore(df, feat_cols).astype(np.float32)
y = df["arousal_class_persubj"].to_numpy(dtype=np.int64)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

seq_X, seq_y, seq_pid = build_sequences(df, X, y, k=K_DEFAULT)
print(f"Последовательностей: {len(seq_X)}, shape={seq_X.shape}")
print(f"Target распределение: {np.bincount(seq_y).tolist()}")

Последовательностей: 12800, shape=(12800, 10, 143)
Target распределение: [7104, 5696]


## 9.2 LSTM — LOSO (32 фолда)

In [3]:
participants = np.unique(seq_pid)
fold_metrics_lstm = []
t_global = time.time()
for i, pid in enumerate(participants, 1):
    test_mask = seq_pid == pid
    train_mask = ~test_mask
    X_tr, y_tr = seq_X[train_mask], seq_y[train_mask]
    X_te, y_te = seq_X[test_mask], seq_y[test_mask]
    if len(np.unique(y_tr)) < 2 or len(X_te) == 0:
        continue
    m = train_fold(X_tr, y_tr, X_te, y_te, bidirectional=False,
                   n_features=seq_X.shape[2])
    fold_metrics_lstm.append(m)
    if i % 8 == 0 or i == len(participants):
        print(f"  [{i:02}/32] {pid:<4}  acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}")

acc = np.array([m['accuracy'] for m in fold_metrics_lstm])
f1 = np.array([m['f1_macro'] for m in fold_metrics_lstm])
auc = np.array([m['auc_roc'] for m in fold_metrics_lstm])
print(f"\nLSTM LOSO:  accuracy = {acc.mean():.4f} ± {acc.std():.4f}")
print(f"            f1_macro = {f1.mean():.4f} ± {f1.std():.4f}")
print(f"            auc_roc  = {np.nanmean(auc):.4f} ± {np.nanstd(auc):.4f}")
print(f"            время:   {time.time()-t_global:.1f} с")

  [08/32] P16   acc=0.5925  f1=0.5713


  [16/32] P23   acc=0.6900  f1=0.6855


  [24/32] P30   acc=0.6100  f1=0.5623


  [32/32] P9    acc=0.6500  f1=0.6305

LSTM LOSO:  accuracy = 0.6115 ± 0.0847
            f1_macro = 0.6007 ± 0.0872
            auc_roc  = 0.6563 ± 0.1116
            время:   97.7 с


## 9.3 BiLSTM — LOSO

In [4]:
fold_metrics_bilstm = []
t_global = time.time()
for i, pid in enumerate(participants, 1):
    test_mask = seq_pid == pid
    train_mask = ~test_mask
    X_tr, y_tr = seq_X[train_mask], seq_y[train_mask]
    X_te, y_te = seq_X[test_mask], seq_y[test_mask]
    if len(np.unique(y_tr)) < 2 or len(X_te) == 0:
        continue
    m = train_fold(X_tr, y_tr, X_te, y_te, bidirectional=True,
                   n_features=seq_X.shape[2])
    fold_metrics_bilstm.append(m)
    if i % 8 == 0 or i == len(participants):
        print(f"  [{i:02}/32] {pid:<4}  acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}")

acc = np.array([m['accuracy'] for m in fold_metrics_bilstm])
f1 = np.array([m['f1_macro'] for m in fold_metrics_bilstm])
auc = np.array([m['auc_roc'] for m in fold_metrics_bilstm])
print(f"\nBiLSTM LOSO: accuracy = {acc.mean():.4f} ± {acc.std():.4f}")
print(f"             f1_macro = {f1.mean():.4f} ± {f1.std():.4f}")
print(f"             auc_roc  = {np.nanmean(auc):.4f} ± {np.nanstd(auc):.4f}")
print(f"             время:   {time.time()-t_global:.1f} с")

  [08/32] P16   acc=0.5750  f1=0.5298


  [16/32] P23   acc=0.6625  f1=0.6545


  [24/32] P30   acc=0.6425  f1=0.6010


  [32/32] P9    acc=0.6100  f1=0.6057

BiLSTM LOSO: accuracy = 0.6123 ± 0.0932
             f1_macro = 0.6003 ± 0.0949
             auc_roc  = 0.6516 ± 0.1161
             время:   165.6 с


## 9.4 Сравнение с Random Forest из эксперимента 2 + сохранение

In [5]:
def stats(rows, name):
    a = np.array([r['accuracy'] for r in rows])
    f = np.array([r['f1_macro'] for r in rows])
    u = np.array([r['auc_roc'] for r in rows])
    return {"model": name, "strategy": "LOSO",
            "accuracy_mean": float(a.mean()), "accuracy_std": float(a.std()),
            "f1_mean": float(f.mean()),       "f1_std": float(f.std()),
            "auc_mean": float(np.nanmean(u)), "auc_std": float(np.nanstd(u)),
            "n_folds": len(rows)}

out_rows = [stats(fold_metrics_lstm, "LSTM"),
            stats(fold_metrics_bilstm, "BiLSTM")]
out_df = pd.DataFrame(out_rows)
out_df.to_csv(config.RESULTS_DIR / "exp06_deep_metrics.csv", index=False)

cmp = pd.DataFrame([
    {"model": "Random Forest (эксп. 2)", "accuracy": 0.6526, "f1_macro": 0.6189, "auc_roc": 0.6945},
    {"model": "LSTM (эксп. 6)",   "accuracy": round(out_rows[0]['accuracy_mean'], 4),
     "f1_macro": round(out_rows[0]['f1_mean'], 4), "auc_roc": round(out_rows[0]['auc_mean'], 4)},
    {"model": "BiLSTM (эксп. 6)", "accuracy": round(out_rows[1]['accuracy_mean'], 4),
     "f1_macro": round(out_rows[1]['f1_mean'], 4), "auc_roc": round(out_rows[1]['auc_mean'], 4)},
])
print(cmp.to_string(index=False))
print(f"\nСохранено: exp06_deep_metrics.csv")

                  model  accuracy  f1_macro  auc_roc
Random Forest (эксп. 2)    0.6526    0.6189   0.6945
         LSTM (эксп. 6)    0.6115    0.6007   0.6563
       BiLSTM (эксп. 6)    0.6123    0.6003   0.6516

Сохранено: exp06_deep_metrics.csv


## 9.5 Выводы

* **LSTM ≈ 0,6115, BiLSTM ≈ 0,6123 — обе ниже Random Forest на ≈4 п.п.**
* BiLSTM не даёт прироста относительно LSTM при удвоенном времени обучения.
* Гипотеза «LSTM лучше RF» **не подтвердилась** на текущей выборке n=32.

# Финальное резюме всего цикла

| Эксп. | Постановка | Лучшая модель | LOSO accuracy | Δ vs baseline |
|------|---|---|---|---|
| 1 | window arousal (глобальный порог) | Random Forest | 0,6322 | ≈ baseline 0,6426 |
| 2 | window arousal (per-subject) | Random Forest | **0,6526** | +1,0 п.п. |
| 3 | **block SSQ > 15** | **XGBoost** | **0,7188** | **+7,6 п.п.** |
| 6 | window arousal (DL) | RF (LSTM/BiLSTM проиграли) | 0,6115–0,6123 | −4,0 п.п. от RF |

**повышение точности предсказания cybersickness ≥ 70% LOSO:** XGBoost block-level — 71,88% accuracy